# Monte Carlo Option Pricing Simulation

This notebook runs **NumPy-vectorized** Monte Carlo simulations on the driver.
No Spark UDFs, no broadcast variables — pure vectorized computation.

| Module | Responsibility |
| --- | --- |
| `config/settings.py` | Table names, export paths, logging |
| `transforms/simulation.py` | 6 vectorized simulation methods (NumPy) |
| `utils/simulation_helpers.py` | Data loading, distribution fitting, run orchestration, write |

**Parameters (widgets):** ticker, strike price (K), simulation period (T days), number of runs

**Input:** `default.yfinance_historical_data`  
**Output:** `default.simulation_results`

In [0]:
%pip install scipy --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os

src_dir = "/Workspace/Users/spyros.louvis@ctpsandbox.com/pub-python-repo/databricks/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from config.settings import FULL_TABLE_NAME, FULL_RESULTS_TABLE_NAME, get_export_path, get_logger
from utils.simulation_helpers import (
    load_historical_data, fit_distributions, run_simulations, results_to_spark, write_results,
)

logger = get_logger("monte_carlo_simulation")
logger.info("Modules loaded successfully.")

In [0]:
# ---------------------------------------------------------------------------
# Widgets (parameterized for Job runs or interactive use)
# ---------------------------------------------------------------------------
dbutils.widgets.text("ticker", "^GSPC", "Ticker Symbol")
dbutils.widgets.text("target_value", "5700", "Strike Price K")
dbutils.widgets.text("period", "10", "Simulation Period T (days)")
dbutils.widgets.text("num_simulations", "1000", "Number of Simulations")

ticker = dbutils.widgets.get("ticker")
K = float(dbutils.widgets.get("target_value"))
T = int(dbutils.widgets.get("period"))
num_runs = int(dbutils.widgets.get("num_simulations"))
alt_weight = 0.1

logger.info(f"Parameters: ticker={ticker}, K={K}, T={T}, runs={num_runs}, alt_weight={alt_weight}")

In [0]:
# Load historical log returns and fit distributions
log_returns, S0 = load_historical_data(spark, FULL_TABLE_NAME, ticker)
dist_params = fit_distributions(log_returns)
logger.info("Data loaded and distributions fitted.")

In [0]:
# Run all 6 simulation methods (NumPy vectorized on driver)
results_pdf = run_simulations(
    log_returns, S0, K, T, num_runs,
    dist_params=dist_params,
    alt_weight=alt_weight,
)
display(spark.createDataFrame(results_pdf))

In [0]:
# Convert to Spark DataFrame with metadata and write
results_df = results_to_spark(spark, results_pdf, ticker, S0)
export_path = get_export_path(ticker)
write_results(results_df, FULL_RESULTS_TABLE_NAME, ticker, export_path)
logger.info(f"[DONE] Simulation complete for ticker={ticker}, K={K}, T={T}, runs={num_runs}")